In [1]:
# Import Pandas for data analysis
import pandas as pd

# Load the prepared dataset from Step 1
df = pd.read_csv("../data/concrete_prepared.csv")

# Group observations by curing age
# Calculate the number of tests and strength statistics for each age
age_summary = (
    df.groupby("age_days")
    .agg(
        sample_count=("strength_mpa", "count"),
        avg_strength=("strength_mpa", "mean"),
        median_strength=("strength_mpa", "median"),
        min_strength=("strength_mpa", "min"),
        max_strength=("strength_mpa", "max")
    )
    .reset_index()
)

# Display the 15 most frequently tested curing ages
# Round the strength values to two decimal places
display(
    age_summary.sort_values(
        "sample_count", ascending=False
    ).head(15).round(2)
)

,age_days,sample_count,avg_strength,median_strength,min_strength,max_strength
4,28,419,36.43,33.73,8.54,81.75
1,3,129,18.38,15.52,2.33,41.64
2,7,122,25.18,21.17,7.51,59.09
5,56,86,50.72,51.15,23.25,80.20
3,14,62,28.75,26.54,12.84,59.76
6,90,54,40.48,39.68,21.86,69.66
8,100,52,47.67,46.98,33.54,66.95
10,180,26,41.73,40.91,24.10,71.62
7,91,17,68.67,67.80,56.50,82.60
13,365,14,43.56,42.82,25.08,56.14


In [2]:
# List the ingredients that define a concrete recipe
# Age is excluded because we're comparing the same recipe at different ages
mix_columns = [
    "cement", "slag", "fly_ash", "water",
    "superplasticizer", "coarse_aggregate", "fine_aggregate"
]

# Calculate the average strength for each recipe at each curing age
# This also handles cases where a recipe was tested more than once at the same age
mix_strength = (
    df.groupby(mix_columns + ["age_days"], as_index=False)
    ["strength_mpa"].mean()
)

# Arrange curing ages into separate columns
# Each row now represents one unique concrete recipe
strength_by_age = mix_strength.pivot(
    index=mix_columns,
    columns="age_days",
    values="strength_mpa"
)

# Keep only recipes tested at both 7 and 28 days
paired_7_28 = strength_by_age[[7, 28]].dropna().copy()

# Rename the columns to make the results easier to understand
paired_7_28.columns = ["strength_7d", "strength_28d"]

# Calculate the strength gained between day 7 and day 28
paired_7_28["strength_gain_mpa"] = (
    paired_7_28["strength_28d"] - paired_7_28["strength_7d"]
)

# Calculate the percentage increase relative to the 7-day strength
paired_7_28["strength_gain_pct"] = (
    paired_7_28["strength_gain_mpa"] /
    paired_7_28["strength_7d"]
) * 100

# Check how many recipes can be compared
print("Recipes tested at both 7 and 28 days:", len(paired_7_28))

# Summarize how much strength these recipes gained
print("\nAverage strength gain (MPa):",
      round(paired_7_28["strength_gain_mpa"].mean(), 2))

print("Median strength gain (MPa):",
      round(paired_7_28["strength_gain_mpa"].median(), 2))

print("Recipes with increased strength:",
      (paired_7_28["strength_gain_mpa"] > 0).sum())

# Display the first 10 recipes and their strength development
display(paired_7_28.reset_index().head(10).round(2))

Recipes tested at both 7 and 28 days: 115

Average strength gain (MPa): 12.87
Median strength gain (MPa): 12.7
Recipes with increased strength: 115


,cement,slag,fly_ash,water,superplasticizer,coarse_aggregate,fine_aggregate,strength_7d,strength_28d,strength_gain_mpa,strength_gain_pct
0,102.0,153.0,0.0,192.0,0.0,887.0,942.0,7.68,17.28,9.60,125.06
1,108.3,162.4,0.0,203.5,0.0,938.2,849.0,7.72,20.59,12.87,166.64
2,116.0,173.0,0.0,192.0,0.0,909.8,891.9,10.09,22.35,12.26,121.49
3,122.6,183.9,0.0,203.5,0.0,958.2,800.1,10.35,24.29,13.94,134.59
4,133.0,200.0,0.0,192.0,0.0,927.4,839.2,13.66,27.87,14.21,104.00
5,135.7,203.5,0.0,185.7,0.0,1076.2,759.3,7.51,18.20,10.69,142.42
6,139.6,209.4,0.0,192.0,0.0,1047.0,806.9,14.59,28.24,13.65,93.55
7,141.3,212.0,0.0,203.5,0.0,971.8,748.5,10.39,29.89,19.50,187.60
8,153.0,102.0,0.0,192.0,0.0,888.0,943.1,8.37,17.96,9.59,114.52
9,157.0,236.0,0.0,192.0,0.0,935.4,781.2,16.89,33.66,16.77,99.32


In [3]:
# Focus on concrete tested at 28 days
# This lets us compare mixtures at the same curing age
df_28 = df[df["age_days"] == 28].copy()

# Average strength measurements for identical recipes
# Each recipe will appear only once in this analysis
df_28 = (
    df_28.groupby(mix_columns, as_index=False)
    .agg(strength_mpa=("strength_mpa", "mean"))
)

# Calculate the water-to-binder ratio for each recipe
df_28["total_binder"] = (
    df_28["cement"] + df_28["slag"] + df_28["fly_ash"]
)

df_28["water_binder_ratio"] = (
    df_28["water"] / df_28["total_binder"]
)

# Calculate the percentage of binder replaced by slag and fly ash
df_28["cement_replacement_pct"] = (
    (df_28["slag"] + df_28["fly_ash"]) /
    df_28["total_binder"]
) * 100

# Select the ingredients and ratios we want to investigate
features = mix_columns + [
    "total_binder",
    "water_binder_ratio",
    "cement_replacement_pct"
]

# Calculate Spearman correlations with compressive strength
# Spearman measures the direction and strength of monotonic relationships
correlations = (
    df_28[features + ["strength_mpa"]]
    .corr(method="spearman")["strength_mpa"]
    .drop("strength_mpa")
)

# Sort by absolute correlation to identify the strongest relationships
correlations = correlations.sort_values(
    key=lambda values: values.abs(),
    ascending=False
)

# Display the results
print("Unique recipes tested at 28 days:", len(df_28))

display(
    correlations.to_frame(
        name="spearman_correlation"
    ).round(3)
)

Unique recipes tested at 28 days: 417


,spearman_correlation
water_binder_ratio,-0.793
total_binder,0.737
cement,0.645
water,-0.343
cement_replacement_pct,-0.289
fly_ash,-0.229
fine_aggregate,-0.190
superplasticizer,0.179
slag,0.156
coarse_aggregate,-0.143


In [4]:
# Divide recipes into three groups based on water-to-binder ratio
# This lets us compare low, medium, and high ratios
df_28["wb_group"] = pd.qcut(
    df_28["water_binder_ratio"],
    q=3,
    labels=["Low", "Medium", "High"]
)

# Divide recipes into three groups based on total binder content
df_28["binder_group"] = pd.qcut(
    df_28["total_binder"],
    q=3,
    labels=["Low", "Medium", "High"]
)

# Compare concrete strength across combinations of both factors
# Each group contains recipes with similar relative ingredient proportions
mix_comparison = (
    df_28.groupby(
        ["wb_group", "binder_group"],
        observed=True
    )
    .agg(
        recipe_count=("strength_mpa", "count"),
        avg_strength=("strength_mpa", "mean"),
        median_strength=("strength_mpa", "median")
    )
    .reset_index()
)

# Display the strength comparison for all nine combinations
display(mix_comparison.round(2))

# Show the actual water-to-binder ratio ranges used in each group
print("\nWater-to-binder ratio ranges:")
display(
    df_28.groupby("wb_group", observed=True)
    ["water_binder_ratio"]
    .agg(["min", "max"])
    .round(3)
)

# Show the binder content ranges used in each group
print("\nTotal binder ranges (kg/m³):")
display(
    df_28.groupby("binder_group", observed=True)
    ["total_binder"]
    .agg(["min", "max"])
    .round(2)
)

,wb_group,binder_group,recipe_count,avg_strength,median_strength
0,Low,Medium,19,49.65,45.94
1,Low,High,120,50.56,50.86
2,Medium,Low,22,25.79,29.00
3,Medium,Medium,98,34.89,35.86
4,Medium,High,19,34.47,35.31
5,High,Low,119,24.01,24.48
6,High,Medium,20,32.31,31.92



Water-to-binder ratio ranges:


,min,max
wb_group,,
Low,0.235,0.402
Medium,0.403,0.509
High,0.509,0.900



Total binder ranges (kg/m³):


,min,max
binder_group,,
Low,200.00,369.64
Medium,371.91,455.00
High,455.10,640.00


In [5]:
# Divide recipes into three groups based on water-to-binder ratio
# This lets us compare low, medium, and high ratios
df_28["wb_group"] = pd.qcut(
    df_28["water_binder_ratio"],
    q=3,
    labels=["Low", "Medium", "High"]
)

# Divide recipes into three groups based on total binder content
df_28["binder_group"] = pd.qcut(
    df_28["total_binder"],
    q=3,
    labels=["Low", "Medium", "High"]
)

# Compare concrete strength across combinations of both factors
# Each group contains recipes with similar relative ingredient proportions
mix_comparison = (
    df_28.groupby(
        ["wb_group", "binder_group"],
        observed=True
    )
    .agg(
        recipe_count=("strength_mpa", "count"),
        avg_strength=("strength_mpa", "mean"),
        median_strength=("strength_mpa", "median")
    )
    .reset_index()
)

# Display the strength comparison for all nine combinations
display(mix_comparison.round(2))

# Show the actual water-to-binder ratio ranges used in each group
print("\nWater-to-binder ratio ranges:")
display(
    df_28.groupby("wb_group", observed=True)
    ["water_binder_ratio"]
    .agg(["min", "max"])
    .round(3)
)

# Show the binder content ranges used in each group
print("\nTotal binder ranges (kg/m³):")
display(
    df_28.groupby("binder_group", observed=True)
    ["total_binder"]
    .agg(["min", "max"])
    .round(2)
)

,wb_group,binder_group,recipe_count,avg_strength,median_strength
0,Low,Medium,19,49.65,45.94
1,Low,High,120,50.56,50.86
2,Medium,Low,22,25.79,29.00
3,Medium,Medium,98,34.89,35.86
4,Medium,High,19,34.47,35.31
5,High,Low,119,24.01,24.48
6,High,Medium,20,32.31,31.92



Water-to-binder ratio ranges:


,min,max
wb_group,,
Low,0.235,0.402
Medium,0.403,0.509
High,0.509,0.900



Total binder ranges (kg/m³):


,min,max
binder_group,,
Low,200.00,369.64
Medium,371.91,455.00
High,455.10,640.00


In [6]:
# Set an example 28-day strength requirement for our analysis
# We'll make this adjustable in the final dashboard
target_strength = 40

# Focus on mixtures with a low water-to-binder ratio
# These showed the strongest average performance in our previous comparison
low_wb = df_28[df_28["wb_group"] == "Low"].copy()

# Check which recipes meet or exceed our example strength target
low_wb["meets_target"] = low_wb["strength_mpa"] >= target_strength

# Compare binder use, water-to-binder ratios, and strength
# This helps us identify which group deserves closer investigation
candidate_summary = (
    low_wb.groupby("binder_group", observed=True)
    .agg(
        recipe_count=("strength_mpa", "count"),
        avg_binder=("total_binder", "mean"),
        avg_wb_ratio=("water_binder_ratio", "mean"),
        avg_strength=("strength_mpa", "mean"),
        median_strength=("strength_mpa", "median"),
        min_strength=("strength_mpa", "min"),
        pct_meeting_target=("meets_target", "mean")
    )
    .reset_index()
)

# Convert the proportion meeting the target into a percentage
candidate_summary["pct_meeting_target"] *= 100

# Display the comparison
print(f"Example strength requirement: {target_strength} MPa at 28 days")

display(candidate_summary.round(2))

Example strength requirement: 40 MPa at 28 days


,binder_group,recipe_count,avg_binder,avg_wb_ratio,avg_strength,median_strength,min_strength,pct_meeting_target
0,Medium,19,417.46,0.38,49.65,45.94,24.24,84.21
1,High,120,516.48,0.34,50.56,50.86,23.89,77.50


In [7]:
# Display the most important results for our business comparison
# This avoids cutting off columns in the notebook

display(
    candidate_summary[
        [
            "binder_group",
            "recipe_count",
            "avg_binder",
            "avg_strength",
            "min_strength",
            "pct_meeting_target"
        ]
    ].round(2)
)

,binder_group,recipe_count,avg_binder,avg_strength,min_strength,pct_meeting_target
0,Medium,19,417.46,49.65,24.24,84.21
1,High,120,516.48,50.56,23.89,77.50


In [8]:
# Set the example strength requirement
target_strength = 40

# Find 28-day recipes that met or exceeded the target
candidates = df_28[
    df_28["strength_mpa"] >= target_strength
].copy()

# Calculate how far each recipe exceeded the target
# This is the observed strength margin, not a guarantee of future performance
candidates["strength_margin_mpa"] = (
    candidates["strength_mpa"] - target_strength
)

# Sort by total binder content, starting with the lowest
# If two recipes use the same binder quantity, show the stronger one first
candidates = candidates.sort_values(
    ["total_binder", "strength_mpa"],
    ascending=[True, False]
)

# Count how many recipes met the requirement
print("Recipes meeting the 40 MPa target:", len(candidates))

# Show the 10 qualifying recipes with the lowest binder content
# Include all ingredient quantities so we can compare actual mixtures
display(
    candidates[
        [
            "cement", "slag", "fly_ash", "water",
            "superplasticizer", "coarse_aggregate",
            "fine_aggregate", "total_binder",
            "water_binder_ratio", "strength_mpa",
            "strength_margin_mpa"
        ]
    ].head(10).round(2)
)

Recipes meeting the 40 MPa target: 139


,cement,slag,fly_ash,water,superplasticizer,coarse_aggregate,fine_aggregate,total_binder,water_binder_ratio,strength_mpa,strength_margin_mpa
331,336.00,0.00,0.00,182.00,3.00,986.00,817.00,336.00,0.54,44.86,4.86
162,213.72,98.05,24.51,181.71,6.86,1065.80,785.38,336.28,0.54,45.71,5.71
164,213.76,98.06,24.52,181.74,6.65,1066.00,785.52,336.34,0.54,40.23,0.23
332,336.50,0.00,0.00,181.90,3.40,985.80,816.80,336.50,0.54,44.87,4.87
202,252.31,0.00,98.75,146.25,14.17,987.76,889.01,351.06,0.42,50.60,10.60
188,246.83,0.00,125.08,143.30,11.99,1086.80,800.89,371.91,0.39,52.50,12.50
232,277.05,0.00,97.39,160.60,11.83,973.90,875.61,374.44,0.43,48.28,8.28
274,304.00,76.00,0.00,228.00,0.00,932.00,670.00,380.00,0.60,47.81,7.81
217,266.00,114.00,0.00,228.00,0.00,932.00,670.00,380.00,0.60,45.85,5.85
139,190.00,190.00,0.00,228.00,0.00,932.00,670.00,380.00,0.60,40.86,0.86


In [9]:
# Compare how many 28-day recipes achieve different strength levels
# This helps us understand the trade-off between strength and binder use

for strength_target in [40, 45, 50]:

    # Find recipes that meet the selected strength level
    eligible = df_28[
        df_28["strength_mpa"] >= strength_target
    ]

    # Find the lowest binder quantity among qualifying recipes
    min_binder = eligible["total_binder"].min()

    print(f"\nStrength target: {strength_target} MPa")
    print("Qualifying recipes:", len(eligible))
    print("Lowest binder content:", round(min_binder, 2), "kg/m³")


# Look more closely at recipes achieving at least 45 MPa
# This gives us a 5 MPa observed margin above our example 40 MPa target
shortlist = df_28[
    df_28["strength_mpa"] >= 45
].copy()

# Calculate the observed strength margin above 40 MPa
shortlist["strength_margin"] = shortlist["strength_mpa"] - 40

# Start with recipes using the least total binder
shortlist = shortlist.sort_values(
    ["total_binder", "strength_mpa"],
    ascending=[True, False]
)

# Display the first 10 candidates with their key characteristics
display(
    shortlist[
        [
            "cement", "slag", "fly_ash", "water",
            "total_binder", "water_binder_ratio",
            "strength_mpa", "strength_margin"
        ]
    ].head(10).round(2)
)


Strength target: 40 MPa
Qualifying recipes: 139
Lowest binder content: 336.0 kg/m³

Strength target: 45 MPa
Qualifying recipes: 92
Lowest binder content: 336.28 kg/m³

Strength target: 50 MPa
Qualifying recipes: 71
Lowest binder content: 351.06 kg/m³


,cement,slag,fly_ash,water,total_binder,water_binder_ratio,strength_mpa,strength_margin
162,213.72,98.05,24.51,181.71,336.28,0.54,45.71,5.71
202,252.31,0.00,98.75,146.25,351.06,0.42,50.60,10.60
188,246.83,0.00,125.08,143.30,371.91,0.39,52.50,12.50
232,277.05,0.00,97.39,160.60,374.44,0.43,48.28,8.28
274,304.00,76.00,0.00,228.00,380.00,0.60,47.81,7.81
217,266.00,114.00,0.00,228.00,380.00,0.60,45.85,5.85
160,213.50,0.00,174.24,154.61,387.74,0.40,45.94,5.94
165,214.90,53.80,121.89,155.63,390.59,0.40,52.20,12.20
226,275.07,0.00,121.35,159.48,396.42,0.40,51.33,11.33
166,218.23,54.64,123.78,140.75,396.65,0.35,55.51,15.51
